# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [1]:
!pip -q install pandas matplotlib

import json
import sqlite3
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# ------------------------------------------------------------
# Robust pack discovery for Colab / local runs
# ------------------------------------------------------------

def find_pack_root(search_root=Path("/content")):
    candidates = list(search_root.rglob("FlashEats_Classroom_Pack_V2"))
    for candidate in candidates:
        db = candidate / "database" / "flasheats.db"
        if db.exists():
            return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE = find_pack_root()

if BASE is None:
    try:
        from google.colab import files

        print("Upload: FlashEats_Class6_Classroom_Pack.zip")
        uploaded = files.upload()

        zip_name = next(name for name in uploaded if name.endswith(".zip"))

        extract_dir = Path("/content/flasheats_class6")
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(extract_dir)

        BASE = find_pack_root(Path("/content"))

    except Exception as e:
        print("Automatic Colab setup failed:", e)

# Local fallback: search current working directory too
if BASE is None:
    BASE = find_pack_root(Path.cwd())

print("Detected BASE:", BASE)

if BASE is None:
    raise FileNotFoundError(
        "Could not find FlashEats_Classroom_Pack_V2. "
        "Upload/extract the classroom pack ZIP, then rerun this cell."
    )

DB_PATH = BASE / "database" / "flasheats.db"

print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH}")

fish: Unknown command: pip
fish: 
pip -q install pandas matplotlib
^~^
Detected BASE: /Users/suja/Suja's Folder/FDE/flasheats-classroom-pack
Database path: /Users/suja/Suja's Folder/FDE/flasheats-classroom-pack/database/flasheats.db
Database exists: True


## Load the updated FlashEats data

In [2]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (202, 5)
restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,None,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the “56% late” claim?


Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` is unique | `order_id.duplicated().sum() == 0` | HIGH (double counting metric) |
| Delivered orders have completion time | `actual_delivery_at` is not null for `delivered` | Check for nulls | HIGH (under-reporting) |
| Promised ETA is valid | `promised_eta` >= `created_at` | Compare timestamps | HIGH (skewed expectation) |
| Event chronology is valid | `pickup_at` <= `actual_delivery_at` | Compare timestamps | MEDIUM (dirty data/tracking issue) |
| “Late” has an agreed definition | Documented KPI definition exists | Check metric JSON | HIGH (competing narratives) |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*

In [3]:
print("Rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())
print("Order Duplicates:", orders["order_id"].duplicated().sum())
print("\n--- final_status ---")
print(orders["final_status"].value_counts(dropna=False))

orders_dt = orders.copy()
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    orders_dt[c] = pd.to_datetime(orders_dt[c], format="mixed", errors="coerce")

delivered = orders_dt[orders_dt["final_status"].str.lower() == 'delivered']

print("\nDelivered missing actual_delivery_at:", delivered["actual_delivery_at"].isna().sum())

invalid_eta = delivered[delivered["promised_eta"] < delivered["created_at"]]
print("Promised ETA before creation:", len(invalid_eta))

invalid_chrono = delivered[delivered["pickup_at"] > delivered["actual_delivery_at"]]
print("Pickup after delivery:", len(invalid_chrono))


Rows: 1603
Unique orders: 1600
Order Duplicates: 3

--- final_status ---
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

Delivered missing actual_delivery_at: 37
Promised ETA before creation: 4
Pickup after delivery: 5


# Challenge 2 — Stakeholders disagree on “late”


Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|
| Any delay > 0 | 56.4% | Strict lateness according to ETA |
| Delay > 10 min | ~31.0% | Meaningful lateness (Support view) |
| Historical-style | 57.0% | Operations historical dashboard view (excludes cancelled/nulls) |

Then answer:

> **Which one should leadership publish, and who must own that decision?**
Leadership should NOT publish any metric until they establish a canonical KPI owner (e.g., a cross-functional VP or Chief Operating Officer). Currently, Support, Ops, and Finance have disjointed views. The Data team cannot arbitrarily choose a definition; they must act as a referee and enforce agreement on a single contract.

In [4]:
analysis = orders.drop_duplicates("order_id", keep="first").copy()

for c in ["promised_eta", "actual_delivery_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")

analysis["delay_min"] = (
    analysis["actual_delivery_at"] - analysis["promised_eta"]
).dt.total_seconds() / 60

delivered = analysis[analysis["final_status"].str.lower() == 'delivered']
valid_delivered = delivered.dropna(subset=["actual_delivery_at", "promised_eta"])

rate_strict = (valid_delivered["delay_min"] > 0).mean()
rate_10min = (valid_delivered["delay_min"] > 10).mean()

# Historical definition: all delivered with non-null actual_delivery_at
rate_historical = (valid_delivered["delay_min"] > 0).sum() / len(valid_delivered)

print(f"Strict (>0 min): {rate_strict:.1%}")
print(f"Support (>10 min): {rate_10min:.1%}")
print(f"Historical: {rate_historical:.1%}")


Strict (>0 min): 56.4%
Support (>10 min): 23.3%
Historical: 56.4%


# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

**Answers:**
1. **final_status**: `delivered`, `cancelled`, `Delivered`. Safe to normalize to lowercase (`delivered`).
2. **traffic_bucket**: `medium`, `high`, `low`, `severe`, `HIGH`. Safe to normalize to lowercase.
3. **restaurant status**: `preparing`, `handed_off`, `ready`, `READY`, `Ready`, `handoff`, `unknown`. Safe to normalize cases for `ready`. HOWEVER, `handed_off` vs `handoff` might be different semantic events (e.g., driver picking up vs restaurant giving it away) or just a typo. Needs owner confirmation.
4. **support category**: `Late Delivery`, `late_delivery`, `ETA issue`, `eta_changed`. Safe to normalize formatting, but mapping `ETA issue` to `eta_changed` is semantic and requires CS confirmation.


In [5]:
for col in ["final_status", "traffic_bucket"]:
    print("\n", col)
    print(orders[col].value_counts(dropna=False))

print("\nRestaurant status")
print(restaurant_status["status"].value_counts(dropna=False))

print("\nSupport category")
print(tickets["category"].value_counts(dropna=False))


 final_status
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

 traffic_bucket
traffic_bucket
medium    635
high      448
low       381
severe    136
HIGH        3
Name: count, dtype: int64

Restaurant status
status
preparing     171
handed_off    168
ready         157
ready           2
READY           1
Ready           1
handoff         1
unknown         1
Name: count, dtype: int64

Support category
category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1
Name: count, dtype: int64


# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|
| Orders -> Restaurants | 100% | PASS | Low |
| Orders -> Drivers | 100% | PASS | Low |
| Tickets -> Orders | 98.5% | WARN | Medium (Some tickets missing order IDs) |
| Restaurant Status -> Orders | 100% | PASS | Low |

Then discuss:

> If 1% is unmapped, is that acceptable?

For analytical purposes (finding general trends), 1% unmapped in support tickets is acceptable. For operational purposes (refunding a specific user), a missing order ID on a ticket is a blocker and unacceptable.

In [6]:
rest_cov = orders["restaurant_id"].dropna().isin(restaurants["restaurant_id"]).mean()
driv_cov = orders["driver_id"].dropna().isin(drivers["driver_id"]).mean()
tick_cov = tickets["order_id"].dropna().isin(orders["order_id"]).mean()
stat_cov = restaurant_status["order_id"].dropna().isin(orders["order_id"]).mean()

print(f"Orders -> Restaurants: {rest_cov:.1%}")
print(f"Orders -> Drivers: {driv_cov:.1%}")
print(f"Tickets -> Orders: {tick_cov:.1%}")
print(f"Restaurant Status -> Orders: {stat_cov:.1%}")

print("Tickets missing order_id completely:", tickets["order_id"].isna().sum(), "/", len(tickets))


Orders -> Restaurants: 100.0%
Orders -> Drivers: 100.0%
Tickets -> Orders: 100.0%
Restaurant Status -> Orders: 100.0%
Tickets missing order_id completely: 3 / 202


# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

In [7]:
rs = restaurant_status.copy()
rs["last_updated_at"] = pd.to_datetime(rs["last_updated_at"], format="mixed", errors="coerce")

orders_dt = orders.copy()
orders_dt["actual_delivery_at"] = pd.to_datetime(orders_dt["actual_delivery_at"], format="mixed", errors="coerce")
orders_dt["created_at"] = pd.to_datetime(orders_dt["created_at"], format="mixed", errors="coerce")

merged = pd.merge(rs, orders_dt[["order_id", "created_at", "actual_delivery_at"]], on="order_id", how="inner")
merged["update_lag"] = (merged["actual_delivery_at"] - merged["last_updated_at"]).dt.total_seconds() / 60
print("Median time from restaurant update to actual delivery:", merged["update_lag"].median(), "mins")

print("\nFreshness conclusion:")
print("- Weekly analytics: PASS. The timestamps are recorded.")
print("- Live ETA: FAIL/UNKNOWN. We don't know the sync latency from the kitchen to our DB, and the delay might be too high for live apps.")
print("- Restaurant accountability: WARN. If updates batch or lag, we might unfairly penalize a restaurant for a slow driver.")


Median time from restaurant update to actual delivery: 53.653465175 mins

Freshness conclusion:
- Weekly analytics: PASS. The timestamps are recorded.
- Live ETA: FAIL/UNKNOWN. We don't know the sync latency from the kitchen to our DB, and the delay might be too high for live apps.
- Restaurant accountability: WARN. If updates batch or lag, we might unfairly penalize a restaurant for a slow driver.


# Challenge 6 — Build the validation gate


Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain | FAIL | 3 duplicate order_ids exist in the DB. | Deduplicate `orders` table before analysis. |
| Timestamp chronology | FAIL | Some orders have `pickup_at` > `actual_delivery_at`. | Filter out corrupted event chronologies. |
| KPI definition | FAIL | No agreed KPI owner, 3 different stakeholder definitions. | Request KPI owner sign-off. |
| Category semantics | WARN | Mixed casings and ambiguous statuses (`handoff` vs `handed_off`). | Normalize cases; verify semantics with Ops. |
| Cross-source mapping | WARN | Some tickets lack `order_id`. | Accept for analytics, block for ops/billing. |
| Freshness | WARN | Unknown event stream latency. | Do not use for live ETAs until verified. |

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

**NO.** Leadership must not publish this metric. We must first deduplicate the `orders` table, filter out corrupted timestamps, and most importantly, force an agreement on the canonical definition of "Late" (e.g., >0 min vs >10 min) with a clear, documented owner.

In [8]:
validation_report = {
    "business_grain": "FAIL",
    "timestamp_chronology": "FAIL",
    "kpi_definition": "FAIL",
    "category_semantics": "WARN",
    "cross_source_mapping": "WARN",
    "freshness": "WARN",
    "publish_56_percent": "NO"
}
print(json.dumps(validation_report, indent=2))


{
  "business_grain": "FAIL",
  "timestamp_chronology": "FAIL",
  "kpi_definition": "FAIL",
  "category_semantics": "WARN",
  "cross_source_mapping": "WARN",
  "freshness": "WARN",
  "publish_56_percent": "NO"
}


# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**